<a href="https://colab.research.google.com/github/Supan-Roy/financial-scam-analysis/blob/main/notebooks/07_source_dependency_and_generalization_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Source Dependency and Generalization Analysis

## Research Objective

This notebook investigates whether the exceptionally high performance of the DistilBERT binary financial scam detection model is influenced by dataset source dependency, superficial textual characteristics, or source-specific patterns.

The investigation aims to distinguish genuine semantic learning from potential dataset shortcuts and assess the model's ability to generalize beyond the sources represented in the original corpus.

## Experimental Objectives

The investigation consists of four stages:

### 1. Source Identification

Determine whether the original dataset source can be predicted using message text alone.

### 2. Surface-Feature Baseline

Evaluate how effectively non-semantic structural and statistical text characteristics can predict the binary scam/legitimate labels.

### 3. Controlled Normalization

Apply controlled text normalization techniques and evaluate their effects on the existing fine-tuned DistilBERT model.

### 4. External-Source Evaluation

Assess the model's generalization capability using independently sourced financial scam and legitimate communications, provided both classes are adequately represented.

## Research Context

The original binary classification corpus contains 76,193 real records:

- 13,861 financial scam messages.
- 62,332 legitimate messages.

The corpus was divided using Stratified Group-Aware Greedy Partitioning (SGGP), with near-duplicate clusters kept within individual partitions.

The original DistilBERT experiment achieved:

- Test Accuracy: 99.8950%
- Test Macro F1: 99.8238%
- Scam Precision: 99.4968%
- Scam Recall: 99.9278%
- PR-AUC: 99.9894%

However, scam and legitimate records originate predominantly from different datasets and sources. This creates a potential source-related confounding problem that requires further investigation.

## Experimental Principles

- Preserve the original dataset and its benchmark splits.
- Do not modify or overwrite the original Notebook 06 results.
- Do not use the test set for model fitting or parameter selection.
- Clearly distinguish diagnostic evidence from proof of model behavior.
- Report negative and inconclusive findings honestly.
- Avoid claiming external generalization without an independent evaluation dataset containing both classes.
- Preserve reproducibility through fixed random seeds and documented configurations.

## Expected Outcome

Determine the extent to which source-related characteristics and superficial textual patterns may influence binary classification performance, and establish a more defensible understanding of the model's generalization capability.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the original real-world benchmark partitions for source identification

from pathlib import Path
import pandas as pd
import numpy as np
import random

RANDOM_STATE = 42

BINARY_DATA_PATH = Path(
    "/content/drive/MyDrive/financial-scam-analysis/data/splits"
)

train_df = pd.read_csv(BINARY_DATA_PATH / "train.csv", low_memory=False)
val_df = pd.read_csv(BINARY_DATA_PATH / "validation.csv", low_memory=False)
test_df = pd.read_csv(BINARY_DATA_PATH / "test.csv", low_memory=False)

print("Dataset partitions loaded successfully.")
print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("\nAvailable columns:")
print(train_df.columns.tolist())

Dataset partitions loaded successfully.
Training: (60957, 32)
Validation: (7618, 32)
Test: (7618, 32)

Available columns:
['record_id', 'original_text', 'normalized_text', 'label', 'channel', 'original_channel', 'is_email_gateway', 'has_gateway_headers', 'source', 'source_url', 'source_license', 'source_record_id', 'is_synthetic', 'scam_subtype', 'fraudulent_intent', 'social_engineering_tactics', 'legitimate_category', 'email_subject', 'email_body', 'duplicate_cluster_id', 'char_count_original', 'word_count_original', 'char_count_normalized', 'word_count_normalized', 'placeholder_count_original', 'placeholder_count_normalized', 'generation_scenario', 'risk_or_safety_note', 'language', 'communication_type', 'group_id', 'split']


In [ ]:
# Examine source distribution across dataset partitions and binary labels

for split_name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{'=' * 20} {split_name} {'=' * 20}")

    print("\nRecords per source:")
    print(df["source"].value_counts(dropna=False))

    print("\nSource versus binary label:")
    print(pd.crosstab(
        df["source"],
        df["label"],
        margins=True,
        dropna=False
    ))


==================== Training ====================

Records per source:
source
MeAJOR (trec5)                                          21842
MeAJOR (trec7)                                          15385
MeAJOR (trec6)                                           8903
Smishing Dataset IMC25 (public user reports)             7674
UCI SMS Spam Collection                                  3599
Nigerian_Fraud.csv                                       2613
smishtank.csv                                             404
Phishing and Legitimate Emails Dataset (Kaggle)           400
Financial scams detection dataset (Mendeley Data V2)      137
Name: count, dtype: int64

Source versus binary label:
label                                                   0      1    All
source                                                                 
Financial scams detection dataset (Mendeley Dat...    137      0    137
MeAJOR (trec5)                                      21842      0  21842
MeAJOR (trec6)     

In [ ]:
# Prepare text and source labels for multiclass source identification

source_train = train_df[["normalized_text", "source"]].copy()
source_val = val_df[["normalized_text", "source"]].copy()
source_test = test_df[["normalized_text", "source"]].copy()

for df in [source_train, source_val, source_test]:
    df.dropna(subset=["normalized_text", "source"], inplace=True)
    df["normalized_text"] = df["normalized_text"].astype(str)

X_source_train = source_train["normalized_text"]
y_source_train = source_train["source"]

X_source_val = source_val["normalized_text"]
y_source_val = source_val["source"]

X_source_test = source_test["normalized_text"]
y_source_test = source_test["source"]

print("Source identification datasets prepared.")
print("Training:", len(source_train))
print("Validation:", len(source_val))
print("Test:", len(source_test))
print("\nNumber of source classes:", y_source_train.nunique())
print("\nTraining source distribution:")
print(y_source_train.value_counts())

Source identification datasets prepared.
Training: 60957
Validation: 7618
Test: 7618

Number of source classes: 9

Training source distribution:
source
MeAJOR (trec5)                                          21842
MeAJOR (trec7)                                          15385
MeAJOR (trec6)                                           8903
Smishing Dataset IMC25 (public user reports)             7674
UCI SMS Spam Collection                                  3599
Nigerian_Fraud.csv                                       2613
smishtank.csv                                             404
Phishing and Legitimate Emails Dataset (Kaggle)           400
Financial scams detection dataset (Mendeley Data V2)      137
Name: count, dtype: int64


In [ ]:
# Train a text-only classifier to identify the dataset source

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report
)

source_classifier = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=150000,
            sublinear_tf=True,
            strip_accents="unicode"
        )
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

source_classifier.fit(X_source_train, y_source_train)

val_source_predictions = source_classifier.predict(X_source_val)

print("Source identification — validation results")
print("Accuracy:", round(accuracy_score(y_source_val, val_source_predictions), 4))
print("Macro F1:", round(f1_score(y_source_val, val_source_predictions, average="macro"), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_source_val, val_source_predictions), 4))

print("\nValidation classification report:")
print(classification_report(
    y_source_val,
    val_source_predictions,
    zero_division=0
))

Source identification — validation results
Accuracy: 0.9764
Macro F1: 0.9116
Balanced Accuracy: 0.9034

Validation classification report:
                                                      precision    recall  f1-score   support

Financial scams detection dataset (Mendeley Data V2)       1.00      0.65      0.79        17
                                      MeAJOR (trec5)       0.98      0.98      0.98      2722
                                      MeAJOR (trec6)       0.98      0.96      0.97      1111
                                      MeAJOR (trec7)       1.00      0.99      0.99      1933
                                  Nigerian_Fraud.csv       1.00      0.99      0.99       326
     Phishing and Legitimate Emails Dataset (Kaggle)       1.00      1.00      1.00        50
        Smishing Dataset IMC25 (public user reports)       0.97      0.97      0.97       963
                             UCI SMS Spam Collection       0.91      0.99      0.95       450
               

In [ ]:
# Evaluate source identification on the held-out test partition

test_source_predictions = source_classifier.predict(X_source_test)

print("Source identification — test results")
print("Accuracy:", round(accuracy_score(y_source_test, test_source_predictions), 4))
print("Macro F1:", round(f1_score(y_source_test, test_source_predictions, average="macro"), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_source_test, test_source_predictions), 4))

print("\nTest classification report:")
print(classification_report(
    y_source_test,
    test_source_predictions,
    zero_division=0
))

Source identification — test results
Accuracy: 0.9783
Macro F1: 0.9001
Balanced Accuracy: 0.8907

Test classification report:
                                                      precision    recall  f1-score   support

Financial scams detection dataset (Mendeley Data V2)       0.92      0.69      0.79        16
                                      MeAJOR (trec5)       0.98      0.99      0.98      2727
                                      MeAJOR (trec6)       0.99      0.96      0.97      1113
                                      MeAJOR (trec7)       0.99      0.99      0.99      1926
                                  Nigerian_Fraud.csv       1.00      0.99      0.99       326
     Phishing and Legitimate Emails Dataset (Kaggle)       1.00      1.00      1.00        50
        Smishing Dataset IMC25 (public user reports)       0.97      0.97      0.97       963
                             UCI SMS Spam Collection       0.92      1.00      0.96       451
                           

In [ ]:
# Extract simple text-surface features for binary scam detection

import re
import string
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

def extract_surface_features(text):
    text = str(text)
    words = re.findall(r"\b\w+\b", text)
    letters = [char for char in text if char.isalpha()]
    non_whitespace = [char for char in text if not char.isspace()]

    return {
        "char_count": len(text),
        "word_count": len(words),
        "avg_word_length": np.mean([len(word) for word in words]) if words else 0,
        "uppercase_ratio": sum(c.isupper() for c in letters) / max(len(letters), 1),
        "digit_ratio": sum(c.isdigit() for c in text) / max(len(text), 1),
        "punctuation_ratio": sum(c in string.punctuation for c in text) / max(len(text), 1),
        "whitespace_ratio": sum(c.isspace() for c in text) / max(len(text), 1),
        "url_count": len(re.findall(r"https?://|www\.", text, re.I)),
        "email_count": len(re.findall(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", text)),
        "currency_count": len(re.findall(r"[$€£¥₹৳]", text)),
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),
        "uppercase_word_count": sum(word.isupper() and len(word) > 1 for word in words),
        "line_count": len(text.splitlines()),
        "parentheses_count": text.count("(") + text.count(")"),
        "bracket_count": text.count("[") + text.count("]") + text.count("<") + text.count(">"),
        "colon_count": text.count(":"),
        "semicolon_count": text.count(";"),
        "non_whitespace_char_count": len(non_whitespace)
    }

def build_surface_features(df):
    return pd.DataFrame(
        [extract_surface_features(text) for text in df["original_text"]],
        index=df.index
    ).fillna(0)

X_surface_train = build_surface_features(train_df)
X_surface_val = build_surface_features(val_df)
X_surface_test = build_surface_features(test_df)

y_binary_train = train_df["label"]
y_binary_val = val_df["label"]
y_binary_test = test_df["label"]

print("Surface-feature datasets prepared.")
print("Training:", X_surface_train.shape)
print("Validation:", X_surface_val.shape)
print("Test:", X_surface_test.shape)
print("\nExtracted features:")
print(X_surface_train.columns.tolist())

Surface-feature datasets prepared.
Training: (60957, 19)
Validation: (7618, 19)
Test: (7618, 19)

Extracted features:
['char_count', 'word_count', 'avg_word_length', 'uppercase_ratio', 'digit_ratio', 'punctuation_ratio', 'whitespace_ratio', 'url_count', 'email_count', 'currency_count', 'exclamation_count', 'question_count', 'uppercase_word_count', 'line_count', 'parentheses_count', 'bracket_count', 'colon_count', 'semicolon_count', 'non_whitespace_char_count']


In [ ]:
# Train a standardized surface-feature baseline for binary scam detection

surface_classifier = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    )
])

surface_classifier.fit(X_surface_train, y_binary_train)

val_surface_predictions = surface_classifier.predict(X_surface_val)

print("Surface-feature baseline — validation results")
print("Accuracy:", round(accuracy_score(y_binary_val, val_surface_predictions), 4))
print("Scam Precision:", round(precision_score(y_binary_val, val_surface_predictions), 4))
print("Scam Recall:", round(recall_score(y_binary_val, val_surface_predictions), 4))
print("Scam F1:", round(f1_score(y_binary_val, val_surface_predictions), 4))
print("Macro F1:", round(f1_score(y_binary_val, val_surface_predictions, average="macro"), 4))

print("\nValidation classification report:")
print(classification_report(
    y_binary_val,
    val_surface_predictions,
    target_names=["LEGITIMATE", "SCAM"],
    zero_division=0
))

Surface-feature baseline — validation results
Accuracy: 0.8857
Scam Precision: 0.6228
Scam Recall: 0.9408
Scam F1: 0.7495
Macro F1: 0.8377

Validation classification report:
              precision    recall  f1-score   support

  LEGITIMATE       0.99      0.87      0.93      6233
        SCAM       0.62      0.94      0.75      1385

    accuracy                           0.89      7618
   macro avg       0.80      0.91      0.84      7618
weighted avg       0.92      0.89      0.89      7618



In [ ]:
# Evaluate the surface-feature baseline on the held-out test partition

test_surface_predictions = surface_classifier.predict(X_surface_test)

print("Surface-feature baseline — test results")
print("Accuracy:", round(accuracy_score(y_binary_test, test_surface_predictions), 4))
print("Scam Precision:", round(precision_score(y_binary_test, test_surface_predictions), 4))
print("Scam Recall:", round(recall_score(y_binary_test, test_surface_predictions), 4))
print("Scam F1:", round(f1_score(y_binary_test, test_surface_predictions), 4))
print("Macro F1:", round(f1_score(y_binary_test, test_surface_predictions, average="macro"), 4))

print("\nTest classification report:")
print(classification_report(
    y_binary_test,
    test_surface_predictions,
    target_names=["LEGITIMATE", "SCAM"],
    zero_division=0
))

Surface-feature baseline — test results
Accuracy: 0.8838
Scam Precision: 0.6172
Scam Recall: 0.9509
Scam F1: 0.7485
Macro F1: 0.8365

Test classification report:
              precision    recall  f1-score   support

  LEGITIMATE       0.99      0.87      0.92      6233
        SCAM       0.62      0.95      0.75      1385

    accuracy                           0.88      7618
   macro avg       0.80      0.91      0.84      7618
weighted avg       0.92      0.88      0.89      7618



In [ ]:
# Load the saved DistilBERT model for controlled text perturbation experiments

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = Path(
    "/content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

controlled_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
controlled_model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_PATH)
).to(device)

controlled_model.eval()

print("Saved DistilBERT model loaded successfully.")
print("Device:", device)
print("Model:", MODEL_PATH)
print("Maximum position embeddings:", controlled_model.config.max_position_embeddings)
print("Tokenizer:", controlled_tokenizer.name_or_path)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Saved DistilBERT model loaded successfully.
Device: cuda
Model: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Maximum position embeddings: 512
Tokenizer: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary


In [ ]:
# Create a reproducible stratified sample for controlled normalization experiments

from sklearn.model_selection import train_test_split

controlled_sample = test_df[["original_text", "label", "source"]].copy()

controlled_sample, _ = train_test_split(
    controlled_sample,
    train_size=1000,
    stratify=controlled_sample["label"],
    random_state=RANDOM_STATE
)

controlled_sample = controlled_sample.reset_index(drop=True)

print("Controlled experiment sample prepared.")
print("Sample size:", len(controlled_sample))
print("\nLabel distribution:")
print(controlled_sample["label"].value_counts().sort_index())

print("\nSource distribution:")
print(controlled_sample["source"].value_counts())

Controlled experiment sample prepared.
Sample size: 1000

Label distribution:
label
0    818
1    182
Name: count, dtype: int64

Source distribution:
source
MeAJOR (trec5)                                          359
MeAJOR (trec7)                                          255
MeAJOR (trec6)                                          130
Smishing Dataset IMC25 (public user reports)            122
UCI SMS Spam Collection                                  71
Nigerian_Fraud.csv                                       42
Phishing and Legitimate Emails Dataset (Kaggle)          11
smishtank.csv                                             7
Financial scams detection dataset (Mendeley Data V2)      3
Name: count, dtype: int64


In [ ]:
# Audit the prevalence of URL, email, and financial-message patterns in the test set

pattern_counts = {
    "URL": test_df["original_text"].astype(str).str.contains(
        r"https?://|www\.", case=False, regex=True
    ).sum(),
    "Email address": test_df["original_text"].astype(str).str.contains(
        r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", regex=True
    ).sum(),
    "Angle-bracket placeholders": test_df["original_text"].astype(str).str.contains(
        r"<[^>]+>", regex=True
    ).sum(),
    "Square-bracket placeholders": test_df["original_text"].astype(str).str.contains(
        r"\[[^\]]+\]", regex=True
    ).sum(),
    "Currency symbols": test_df["original_text"].astype(str).str.contains(
        r"[$€£¥₹৳]", regex=True
    ).sum(),
    "Long digit sequences": test_df["original_text"].astype(str).str.contains(
        r"\d{5,}", regex=True
    ).sum()
}

print("Pattern prevalence in the complete test set:")
for pattern, count in pattern_counts.items():
    print(f"{pattern}: {count:,} / {len(test_df):,} ({count / len(test_df):.2%})")

Pattern prevalence in the complete test set:
URL: 0 / 7,618 (0.00%)
Email address: 0 / 7,618 (0.00%)
Angle-bracket placeholders: 3,937 / 7,618 (51.68%)
Square-bracket placeholders: 5,691 / 7,618 (74.70%)
Currency symbols: 909 / 7,618 (11.93%)
Long digit sequences: 682 / 7,618 (8.95%)


In [ ]:
# Identify and count placeholder tokens in the original test messages

from collections import Counter

def extract_placeholders(text):
    text = str(text)

    angle_tokens = re.findall(r"<[^<>\n]{1,100}>", text)
    square_tokens = re.findall(r"\[[^\[\]\n]{1,100}\]", text)

    return angle_tokens, square_tokens

angle_counter = Counter()
square_counter = Counter()

for text in test_df["original_text"].dropna():
    angle_tokens, square_tokens = extract_placeholders(text)

    angle_counter.update(token.strip() for token in angle_tokens)
    square_counter.update(token.strip() for token in square_tokens)

print("Most frequent angle-bracket tokens:")
for token, count in angle_counter.most_common(30):
    print(f"{count:>6}  {token}")

print("\nMost frequent square-bracket tokens:")
for token, count in square_counter.most_common(30):
    print(f"{count:>6}  {token}")

print("\nUnique angle-bracket tokens:", len(angle_counter))
print("Unique square-bracket tokens:", len(square_counter))

Most frequent angle-bracket tokens:
  3202  <|SYMBOL|>
  3051  <|URL|>
  2439  <|EMAIL|>
  1556  <|FILE|>
  1215  <|PHONE|>
   688  <URL>
   554  <DATE_TIME>
   370  <|EMAIL_SEPARATOR|>
   299  <|EMOJI|>
   265  <EMAIL>
   240  <|[[FILE_PATH]]|>
   194  <PHONE>
   193  < File: [FILE_NAME] >
   188  <NAMED_ENTITY>
   136  <PHONE_NUMBER>
   114  <[FILE_NAME]>
   113  <|U[[PRODUCT]]L|>
    96  <|[[URL]]|>
    87  <bmp>
    85  <US_DRIVER_LICENSE>
    74  <LOCATION>
    73  <dv>
    68  <ms% [TIME]>
    62  <ms% 500>
    50  <US_BANK_NUMBER>
    49  <|PGP|>
    39  <br>
    39  <cr>
    38  <[EMAIL_ADDRESS]>
    33  <|[[FILE_NAME]]|>

Most frequent square-bracket tokens:
 19010  [ORGANIZATION]
 17927  [NAME]
 13468  [PRODUCT]
  7643  [DATE]
  5015  [FINANCIAL_INFO]
  4757  [REFERENCE_NUMBER]
  4467  [ADDRESS]
  3046  [TIME]
  2976  [URL]
  2883  [EMAIL_ADDRESS]
  2675  [PHONE_NUMBER]
  1901  [FILE_NAME]
  1378  [USERNAME]
  1252  [FILE_PATH]
  1066  [INITIALS]
   546  [R]
   286  [NUMBER]


In [ ]:
# Compare placeholder formatting patterns across sources and binary labels

def placeholder_record_features(text):
    angle_tokens, square_tokens = extract_placeholders(text)

    return {
        "angle_count": len(angle_tokens),
        "square_count": len(square_tokens),
        "has_angle": int(len(angle_tokens) > 0),
        "has_square": int(len(square_tokens) > 0)
    }

placeholder_features = pd.DataFrame(
    [
        placeholder_record_features(text)
        for text in test_df["original_text"]
    ],
    index=test_df.index
)

placeholder_analysis = pd.concat(
    [
        test_df[["source", "label"]],
        placeholder_features
    ],
    axis=1
)

source_placeholder_summary = (
    placeholder_analysis
    .groupby("source")
    .agg(
        records=("label", "size"),
        scam_rate=("label", "mean"),
        angle_records=("has_angle", "sum"),
        square_records=("has_square", "sum"),
        avg_angle_tokens=("angle_count", "mean"),
        avg_square_tokens=("square_count", "mean")
    )
)

source_placeholder_summary["angle_record_pct"] = (
    source_placeholder_summary["angle_records"]
    / source_placeholder_summary["records"] * 100
)

source_placeholder_summary["square_record_pct"] = (
    source_placeholder_summary["square_records"]
    / source_placeholder_summary["records"] * 100
)

print("Placeholder formatting by source:")
print(source_placeholder_summary.round(2).to_string())

print("\nPlaceholder formatting by binary label:")
print(
    placeholder_analysis.groupby("label")[
        ["has_angle", "has_square", "angle_count", "square_count"]
    ].agg(["mean", "median", "sum"]).round(3)
)

Placeholder formatting by source:
                                                      records  scam_rate  angle_records  square_records  avg_angle_tokens  avg_square_tokens  angle_record_pct  square_record_pct
source                                                                                                                                                                           
Financial scams detection dataset (Mendeley Data V2)       16        0.0              0               0              0.00               0.00              0.00               0.00
MeAJOR (trec5)                                           2727        0.0            977            2647              1.12              13.79             35.83              97.07
MeAJOR (trec6)                                           1113        0.0            486            1066              3.59              11.92             43.67              95.78
MeAJOR (trec7)                                           1926        0.0    

In [ ]:
# Measure how well placeholder formatting alone predicts the binary label

def build_placeholder_features(df):
    features = [
        placeholder_record_features(text)
        for text in df["original_text"]
    ]

    return pd.DataFrame(features, index=df.index)

X_placeholder_train = build_placeholder_features(train_df)
X_placeholder_val = build_placeholder_features(val_df)
X_placeholder_test = build_placeholder_features(test_df)

placeholder_classifier = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    )
])

placeholder_classifier.fit(X_placeholder_train, y_binary_train)

val_placeholder_predictions = placeholder_classifier.predict(X_placeholder_val)

print("Placeholder-only baseline — validation results")
print("Accuracy:", round(accuracy_score(y_binary_val, val_placeholder_predictions), 4))
print("Scam Precision:", round(precision_score(y_binary_val, val_placeholder_predictions), 4))
print("Scam Recall:", round(recall_score(y_binary_val, val_placeholder_predictions), 4))
print("Scam F1:", round(f1_score(y_binary_val, val_placeholder_predictions), 4))
print("Macro F1:", round(f1_score(y_binary_val, val_placeholder_predictions, average="macro"), 4))

print("\nValidation classification report:")
print(classification_report(
    y_binary_val,
    val_placeholder_predictions,
    target_names=["LEGITIMATE", "SCAM"],
    zero_division=0
))

Placeholder-only baseline — validation results
Accuracy: 0.9111
Scam Precision: 0.6749
Scam Recall: 0.9863
Scam F1: 0.8014
Macro F1: 0.8721

Validation classification report:
              precision    recall  f1-score   support

  LEGITIMATE       1.00      0.89      0.94      6233
        SCAM       0.67      0.99      0.80      1385

    accuracy                           0.91      7618
   macro avg       0.84      0.94      0.87      7618
weighted avg       0.94      0.91      0.92      7618



In [ ]:
# Compare original and normalized text to inspect the preprocessing effect

comparison_sample = test_df[
    ["source", "label", "original_text", "normalized_text"]
].sample(10, random_state=RANDOM_STATE)

for index, row in comparison_sample.iterrows():
    print("=" * 100)
    print("Source:", row["source"])
    print("Label:", row["label"])

    print("\nORIGINAL TEXT:")
    print(str(row["original_text"])[:700])

    print("\nNORMALIZED TEXT:")
    print(str(row["normalized_text"])[:700])
    print()

Source: Smishing Dataset IMC25 (public user reports)
Label: 1

ORIGINAL TEXT:
Good <DATE_TIME> <NAMED_ENTITY>. Apologies for the delayed response. We regret the inconvenience caused with the mix up of SMS sent to you. Unfortunately your account doesn't qualify for a 50% offer what we have is a speed upgrade offer for you account. Kindly top up <LOCATION>. 798 for a speed upgrade. ^EK Dial *502# to get your Zuku Fiber account details

NORMALIZED TEXT:
Good [DATE_TIME] [NAMED_ENTITY]. Apologies for the delayed response. We regret the inconvenience caused with the mix up of SMS sent to you. Unfortunately your account doesn't qualify for a 50% offer what we have is a speed upgrade offer for you account. Kindly top up [LOCATION]. 798 for a speed upgrade. ^EK Dial *502# to get your Zuku Fiber account details

Source: MeAJOR (trec5)
Label: 0

ORIGINAL TEXT:
NA Printing & Writing Papers Statistics - Flash Report for September 2001

The North American Printing & Writing Papers Statistics - Flas

In [ ]:
# Audit placeholder tokens remaining in the normalized text

def extract_normalized_placeholders(text):
    text = str(text)

    angle_tokens = re.findall(r"<[^<>\n]{1,100}>", text)
    square_tokens = re.findall(r"\[[^\[\]\n]{1,100}\]", text)

    return angle_tokens, square_tokens


normalized_angle_counter = Counter()
normalized_square_counter = Counter()

for df in [train_df, val_df, test_df]:
    for text in df["normalized_text"].dropna():
        angle_tokens, square_tokens = extract_normalized_placeholders(text)

        normalized_angle_counter.update(
            token.strip() for token in angle_tokens
        )
        normalized_square_counter.update(
            token.strip() for token in square_tokens
        )

print("Most frequent remaining angle-bracket tokens:")
for token, count in normalized_angle_counter.most_common(30):
    print(f"{count:>8,}  {token}")

print("\nMost frequent remaining square-bracket tokens:")
for token, count in normalized_square_counter.most_common(30):
    print(f"{count:>8,}  {token}")

print("\nUnique angle-bracket tokens:", len(normalized_angle_counter))
print("Unique square-bracket tokens:", len(normalized_square_counter))

Most frequent remaining angle-bracket tokens:
  30,705  <|URL|>
  25,474  <|SYMBOL|>
  23,154  <|EMAIL|>
  17,579  <|FILE|>
  11,785  <|PHONE|>
   3,531  <|EMAIL_SEPARATOR|>
   2,773  <|EMOJI|>
   1,771  <|[SYSTEM_ARTIFACT]|>
   1,310  <|U[PRODUCT]L|>
   1,027  <|[URL]|>
     872  <bmp>
     472  <|PGP|>
     434  <cr>
     427  </a>
     403  <|[EMAIL_ADDRESS]|>
     355  <|[PHONE_NUMBER]|>
     335  <[EMAIL_ADDRESS]>
     303  <ms% 500>
     301  <mailto:[EMAIL_ADDRESS]>
     288  <BR>
     247  <nfb>
     246  <->
     241  <umpr>
     226  <br>
     220  <dv>
     212  </DIV>
     210  <[IDENTIFIER]>
     197  <snip>
     186  <umnr>
     184  <CODESITE>

Most frequent remaining square-bracket tokens:
 206,134  [PERSON_NAME]
 182,646  [ORGANIZATION]
 139,758  [PRODUCT]
 110,038  [DATE_TIME]
  59,448  [IDENTIFIER]
  56,200  [FINANCIAL_DATA]
  42,587  [SYSTEM_ARTIFACT]
  40,831  [LOCATION]
  37,745  [URL]
  30,669  [EMAIL_ADDRESS]
  28,219  [PHONE_NUMBER]
   5,503  [R]
   2,757  [1]


In [ ]:
# Measure remaining placeholder formatting by source and binary label

def normalized_placeholder_features(text):
    angle_tokens, square_tokens = extract_normalized_placeholders(text)

    return {
        "angle_count": len(angle_tokens),
        "square_count": len(square_tokens),
        "has_angle": int(len(angle_tokens) > 0),
        "has_square": int(len(square_tokens) > 0)
    }

normalized_features = pd.concat(
    [
        pd.DataFrame(
            [
                normalized_placeholder_features(text)
                for text in df["normalized_text"]
            ],
            index=df.index
        )
        for df in [train_df, val_df, test_df]
    ]
)

all_binary_data = pd.concat(
    [train_df, val_df, test_df]
)

normalized_placeholder_analysis = pd.concat(
    [
        all_binary_data[["source", "label"]].reset_index(drop=True),
        normalized_features.reset_index(drop=True)
    ],
    axis=1
)

print("Placeholder formatting by source:")
print(
    normalized_placeholder_analysis.groupby("source").agg(
        records=("label", "size"),
        scam_rate=("label", "mean"),
        angle_record_pct=("has_angle", "mean"),
        square_record_pct=("has_square", "mean"),
        avg_angle_tokens=("angle_count", "mean"),
        avg_square_tokens=("square_count", "mean")
    ).assign(
        angle_record_pct=lambda x: x["angle_record_pct"] * 100,
        square_record_pct=lambda x: x["square_record_pct"] * 100
    ).round(2).to_string()
)

print("\nPlaceholder formatting by binary label:")
print(
    normalized_placeholder_analysis.groupby("label")[
        ["has_angle", "has_square", "angle_count", "square_count"]
    ].agg(["mean", "median"]).round(3)
)

Placeholder formatting by source:
                                                      records  scam_rate  angle_record_pct  square_record_pct  avg_angle_tokens  avg_square_tokens
source                                                                                                                                            
Financial scams detection dataset (Mendeley Data V2)      170        0.0              0.00               0.00              0.00               0.00
MeAJOR (trec5)                                          27291        0.0             29.99              96.38              0.99              13.94
MeAJOR (trec6)                                          11127        0.0             43.99              95.30              3.20              12.32
MeAJOR (trec7)                                          19244        0.0             70.73              99.52              4.44              22.23
Nigerian_Fraud.csv                                       3265        1.0            

In [ ]:
# Train a placeholder-only classifier using the actual normalized text

X_norm_placeholder_train = pd.DataFrame(
    [
        normalized_placeholder_features(text)
        for text in train_df["normalized_text"]
    ],
    index=train_df.index
)

X_norm_placeholder_val = pd.DataFrame(
    [
        normalized_placeholder_features(text)
        for text in val_df["normalized_text"]
    ],
    index=val_df.index
)

X_norm_placeholder_test = pd.DataFrame(
    [
        normalized_placeholder_features(text)
        for text in test_df["normalized_text"]
    ],
    index=test_df.index
)

normalized_placeholder_classifier = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    )
])

normalized_placeholder_classifier.fit(
    X_norm_placeholder_train,
    y_binary_train
)

val_norm_placeholder_predictions = (
    normalized_placeholder_classifier.predict(X_norm_placeholder_val)
)

print("Normalized placeholder-only baseline — validation")
print("Accuracy:", round(accuracy_score(y_binary_val, val_norm_placeholder_predictions), 4))
print("Scam Precision:", round(precision_score(y_binary_val, val_norm_placeholder_predictions), 4))
print("Scam Recall:", round(recall_score(y_binary_val, val_norm_placeholder_predictions), 4))
print("Scam F1:", round(f1_score(y_binary_val, val_norm_placeholder_predictions), 4))
print("Macro F1:", round(f1_score(y_binary_val, val_norm_placeholder_predictions, average="macro"), 4))

print("\nClassification report:")
print(classification_report(
    y_binary_val,
    val_norm_placeholder_predictions,
    target_names=["LEGITIMATE", "SCAM"],
    zero_division=0
))

Normalized placeholder-only baseline — validation
Accuracy: 0.8135
Scam Precision: 0.4932
Scam Recall: 0.948
Scam F1: 0.6489
Macro F1: 0.7609

Classification report:
              precision    recall  f1-score   support

  LEGITIMATE       0.99      0.78      0.87      6233
        SCAM       0.49      0.95      0.65      1385

    accuracy                           0.81      7618
   macro avg       0.74      0.87      0.76      7618
weighted avg       0.90      0.81      0.83      7618



In [ ]:
# Evaluate the normalized placeholder-only baseline on the test partition

test_norm_placeholder_predictions = (
    normalized_placeholder_classifier.predict(X_norm_placeholder_test)
)

print("Normalized placeholder-only baseline — test")
print("Accuracy:", round(accuracy_score(y_binary_test, test_norm_placeholder_predictions), 4))
print("Scam Precision:", round(precision_score(y_binary_test, test_norm_placeholder_predictions), 4))
print("Scam Recall:", round(recall_score(y_binary_test, test_norm_placeholder_predictions), 4))
print("Scam F1:", round(f1_score(y_binary_test, test_norm_placeholder_predictions), 4))
print("Macro F1:", round(f1_score(y_binary_test, test_norm_placeholder_predictions, average="macro"), 4))

print("\nClassification report:")
print(classification_report(
    y_binary_test,
    test_norm_placeholder_predictions,
    target_names=["LEGITIMATE", "SCAM"],
    zero_division=0
))

Normalized placeholder-only baseline — test
Accuracy: 0.8215
Scam Precision: 0.5048
Scam Recall: 0.943
Scam F1: 0.6576
Macro F1: 0.7684

Classification report:
              precision    recall  f1-score   support

  LEGITIMATE       0.98      0.79      0.88      6233
        SCAM       0.50      0.94      0.66      1385

    accuracy                           0.82      7618
   macro avg       0.74      0.87      0.77      7618
weighted avg       0.90      0.82      0.84      7618



In [ ]:
# Prepare the same reproducible test sample with the actual Transformer input

controlled_sample = test_df[
    ["normalized_text", "label", "source"]
].copy()

controlled_sample, _ = train_test_split(
    controlled_sample,
    train_size=1000,
    stratify=controlled_sample["label"],
    random_state=RANDOM_STATE
)

controlled_sample = controlled_sample.reset_index(drop=True)

print("Controlled Transformer sample prepared.")
print("Sample size:", len(controlled_sample))
print("\nLabel distribution:")
print(controlled_sample["label"].value_counts().sort_index())

Controlled Transformer sample prepared.
Sample size: 1000

Label distribution:
label
0    818
1    182
Name: count, dtype: int64


In [ ]:
# Standardize placeholder wrappers without changing entity categories

def canonicalize_placeholder_wrapper(match):
    token = match.group(0)

    if token.startswith("<|") and token.endswith("|>"):
        entity = token[2:-2]
        return f"[{entity}]"

    if token.startswith("<") and token.endswith(">"):
        entity = token[1:-1]
        return f"[{entity}]"

    return token


PLACEHOLDER_PATTERN = re.compile(
    r"<\|[^<>\n]{1,80}\|>|<[^<>\n]{1,80}>|\[[^\[\]\n]{1,80}\]"
)

controlled_sample["text_canonicalized"] = (
    controlled_sample["normalized_text"].apply(
        lambda text: PLACEHOLDER_PATTERN.sub(
            canonicalize_placeholder_wrapper,
            str(text)
        )
    )
)

print("Wrapper-only canonicalization completed.")
print(
    "Records changed:",
    (
        controlled_sample["normalized_text"]
        != controlled_sample["text_canonicalized"]
    ).sum()
)

example = controlled_sample[
    controlled_sample["normalized_text"] != controlled_sample["text_canonicalized"]
].iloc[0]

print("\nBefore:")
print(example["normalized_text"][:700])

print("\nAfter:")
print(example["text_canonicalized"][:700])

Wrapper-only canonicalization completed.
Records changed: 339

Before:
[ORGANIZATION].com Alert(tm) Forecast For User|[PERSON_NAME]

Alert [ORGANIZATION] 7-Day Forecast for [LOCATION] Tonight L 55 Clear Tomorrow H 81 Very warm with plenty of sun Tomorrow Night L 57 Clear WEDNESDAY H 73 / L 57 Plenty of sunshine THURSDAY H 69 / L 55 Sunny FRIDAY H 70 / L 55 Mostly sunny SATURDAY H 69 / L 56 Fog to sun SUNDAY H 68 / L 58 Cool with plenty of sunshine Choose another forecast: Enter a zipcode, or a city, state <|SYMBOL|>[ORGANIZATION] All rights reserved.

After:
[ORGANIZATION].com Alert(tm) Forecast For User|[PERSON_NAME]

Alert [ORGANIZATION] 7-Day Forecast for [LOCATION] Tonight L 55 Clear Tomorrow H 81 Very warm with plenty of sun Tomorrow Night L 57 Clear WEDNESDAY H 73 / L 57 Plenty of sunshine THURSDAY H 69 / L 55 Sunny FRIDAY H 70 / L 55 Mostly sunny SATURDAY H 69 / L 56 Fog to sun SUNDAY H 68 / L 58 Cool with plenty of sunshine Choose another forecast: Enter a zipcode, or a city, s

In [ ]:
# Verify that canonicalization changes wrappers without changing entity names

def extract_entity_names(text):
    tokens = re.findall(
        r"<\|([^<>|]+)\|>|<([^<>]+)>|\[([^\[\]]+)\]",
        str(text)
    )

    return [
        next(group for group in token if group)
        for token in tokens
    ]

entity_changes = []

for _, row in controlled_sample.iterrows():
    original_entities = extract_entity_names(row["normalized_text"])
    canonical_entities = extract_entity_names(row["text_canonicalized"])

    if original_entities != canonical_entities:
        entity_changes.append({
            "original": original_entities,
            "canonicalized": canonical_entities
        })

print("Entity-category preservation check")
print("Records checked:", len(controlled_sample))
print("Records with entity-category differences:", len(entity_changes))

if entity_changes:
    print("\nFirst differences:")
    for difference in entity_changes[:5]:
        print(difference)
else:
    print("All extracted entity names are preserved.")

Entity-category preservation check
Records checked: 1000
Records with entity-category differences: 45

First differences:
{'original': ['ORGANIZATION', 'PERSON_NAME', 'ORGANIZATION', ' OLE Object: Paintbrush Picture ', 'ORGANIZATION', 'ORGANIZATION', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'PERSON_NAME', 'PHONE_NUMBER', 'URL'], 'canonicalized': ['ORGANIZATION', 'PERSON_NAME', 'ORGANIZATION', '[ OLE Object: Paintbrush Picture ]', 'ORGANIZATION', 'ORGANIZATION', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'DATE_TIME', 'PERSON_NAME', 'PHONE_NUMBER', 'URL']}
{'original': ['PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'FILE', 'FILE', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'SYSTEM_ARTIFACT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRODUCT', 'PRO

In [ ]:
# Restrict canonicalization to explicitly recognized entity markers

KNOWN_ENTITY_MARKERS = {
    "URL", "EMAIL", "EMAIL_ADDRESS",
    "PHONE", "PHONE_NUMBER",
    "DATE", "DATE_TIME",
    "NAME", "PERSON_NAME", "NAMED_ENTITY",
    "ORGANIZATION", "PRODUCT", "LOCATION",
    "FILE", "FILE_NAME", "FILE_PATH",
    "BANK_NUMBER", "FINANCIAL_INFO", "FINANCIAL_DATA",
    "REFERENCE_NUMBER", "IDENTIFIER", "USERNAME",
    "SYSTEM_ARTIFACT", "SYMBOL", "EMOJI"
}

STRICT_ENTITY_PATTERN = re.compile(
    r"<\|([A-Z][A-Z0-9_]*)\|>|<([A-Z][A-Z0-9_]*)>"
)

def strict_canonicalize(match):
    entity = match.group(1) or match.group(2)

    if entity not in KNOWN_ENTITY_MARKERS:
        return match.group(0)

    return f"[{entity}]"

controlled_sample["text_canonicalized"] = (
    controlled_sample["normalized_text"].apply(
        lambda text: STRICT_ENTITY_PATTERN.sub(
            strict_canonicalize,
            str(text)
        )
    )
)

print("Strict canonicalization completed.")

print(
    "Records changed:",
    (
        controlled_sample["normalized_text"]
        != controlled_sample["text_canonicalized"]
    ).sum()
)

print("\nExample transformations:")

changed = controlled_sample[
    controlled_sample["normalized_text"]
    != controlled_sample["text_canonicalized"]
]

for _, row in changed.head(3).iterrows():
    print("\nBefore:", row["normalized_text"][:400])
    print("After: ", row["text_canonicalized"][:400])

Strict canonicalization completed.
Records changed: 296

Example transformations:

Before: [ORGANIZATION].com Alert(tm) Forecast For User|[PERSON_NAME]

Alert [ORGANIZATION] 7-Day Forecast for [LOCATION] Tonight L 55 Clear Tomorrow H 81 Very warm with plenty of sun Tomorrow Night L 57 Clear WEDNESDAY H 73 / L 57 Plenty of sunshine THURSDAY H 69 / L 55 Sunny FRIDAY H 70 / L 55 Mostly sunny SATURDAY H 69 / L 56 Fog to sun SUNDAY H 68 / L 58 Cool with plenty of sunshine Choose another fore
After:  [ORGANIZATION].com Alert(tm) Forecast For User|[PERSON_NAME]

Alert [ORGANIZATION] 7-Day Forecast for [LOCATION] Tonight L 55 Clear Tomorrow H 81 Very warm with plenty of sun Tomorrow Night L 57 Clear WEDNESDAY H 73 / L 57 Plenty of sunshine THURSDAY H 69 / L 55 Sunny FRIDAY H 70 / L 55 Mostly sunny SATURDAY H 69 / L 56 Fog to sun SUNDAY H 68 / L 58 Cool with plenty of sunshine Choose another fore

Before: RE: [ORGANIZATION]

Should we consider the backup childcare as an emergency measure? What 

In [ ]:
# Verify that only allowlisted entity-marker wrappers were changed

verification_errors = []

for _, row in controlled_sample.iterrows():
    expected = STRICT_ENTITY_PATTERN.sub(
        strict_canonicalize,
        str(row["normalized_text"])
    )

    actual = row["text_canonicalized"]

    if expected != actual:
        verification_errors.append(row.name)

print("Transformation integrity check")
print("Records checked:", len(controlled_sample))
print("Verification mismatches:", len(verification_errors))

if not verification_errors:
    print("PASS: All transformed texts match the defined wrapper-only rule.")
else:
    print("FAIL: Unexpected differences found.")
    print("Affected record indices:", verification_errors[:10])

Transformation integrity check
Records checked: 1000
Verification mismatches: 0
PASS: All transformed texts match the defined wrapper-only rule.


In [ ]:
# Compare DistilBERT predictions on original and canonicalized text

def predict_with_transformer(texts, batch_size=16):
    all_probabilities = []

    controlled_model.eval()

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]

            encoded = controlled_tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)

            outputs = controlled_model(**encoded)
            probabilities = torch.softmax(outputs.logits, dim=-1)

            all_probabilities.append(probabilities.cpu().numpy())

    return np.concatenate(all_probabilities, axis=0)


original_probabilities = predict_with_transformer(
    controlled_sample["normalized_text"].tolist()
)

canonical_probabilities = predict_with_transformer(
    controlled_sample["text_canonicalized"].tolist()
)

controlled_sample["original_prediction"] = original_probabilities.argmax(axis=1)
controlled_sample["canonical_prediction"] = canonical_probabilities.argmax(axis=1)

controlled_sample["original_scam_probability"] = original_probabilities[:, 1]
controlled_sample["canonical_scam_probability"] = canonical_probabilities[:, 1]

print("Transformer inference completed.")
print("Original accuracy:",
      round(accuracy_score(controlled_sample["label"],
                           controlled_sample["original_prediction"]), 4))
print("Canonicalized accuracy:",
      round(accuracy_score(controlled_sample["label"],
                           controlled_sample["canonical_prediction"]), 4))

print("Prediction changes:",
      (controlled_sample["original_prediction"]
       != controlled_sample["canonical_prediction"]).sum())

print("Mean absolute scam-probability change:",
      round(np.abs(
          controlled_sample["original_scam_probability"]
          - controlled_sample["canonical_scam_probability"]
      ).mean(), 6))

Transformer inference completed.
Original accuracy: 0.999
Canonicalized accuracy: 0.999
Prediction changes: 0
Mean absolute scam-probability change: 0.0



## Summary, Findings, and Limitations

### Key Findings

This notebook investigated whether the exceptionally strong binary classification performance observed in Notebook 06 could be influenced by source-specific characteristics and dataset composition.

The following findings were obtained:

1. **Source composition:** All nine data sources were represented across the training, validation, and test partitions. However, each source contained examples from only one class. This reflects the natural composition of the collected datasets: some sources consist of legitimate communications, while others consist of fraudulent communications.

2. **Source identification from text:** A TF-IDF and LinearSVC classifier trained exclusively on normalized message text achieved:
   - Validation accuracy: 97.64%
   - Test accuracy: 97.83%
   - Test macro F1: 90.01%

   This demonstrates that source identity is highly recoverable from textual characteristics alone. It does not establish that the DistilBERT binary classifier directly relies on source identity.

3. **Surface-feature baseline:** A Logistic Regression classifier using 19 structural and statistical text features achieved:
   - Validation accuracy: 88.57%
   - Test accuracy: 88.38%
   - Test macro F1: 83.65%

   These results indicate that simple structural characteristics provide useful classification signals but do not independently reproduce the performance of the Transformer model.

4. **Placeholder analysis:** Placeholder distributions differed substantially across sources and classes. In particular, the legitimate corpus contained considerably more square-bracket entity placeholders, while several scam sources contained different placeholder conventions.

   A placeholder-only Logistic Regression classifier using normalized-text features achieved:
   - Validation accuracy: 81.35%
   - Test accuracy: 82.15%
   - Test macro F1: 76.84%

   Placeholder characteristics therefore contain class-associated information, but they are insufficient to explain the near-perfect DistilBERT results.

5. **Controlled placeholder-wrapper experiment:** A controlled sample of 1,000 test records was used to examine the effect of a restricted placeholder-wrapper canonicalization. The transformation changed 296 records while preserving the recognized entity names.

   DistilBERT produced identical classification accuracy (99.9%) and no prediction changes or measurable mean absolute scam-probability changes on this sample.

   This result applies only to the particular transformation and sample evaluated. It does not establish that all placeholder conventions or other source-specific artifacts are irrelevant to the model.

### Interpretation

The investigation establishes that the assembled corpus contains recoverable source-specific textual characteristics and that some of these characteristics are associated with the binary class distribution.

However, source-specific language should not automatically be treated as undesirable leakage. Financial scams naturally exhibit recognizable communication patterns, including urgency, impersonation, fabricated authority, financial incentives, threats, credential requests, and suspicious payment instructions. Learning such patterns is part of the intended detection task.

It is therefore important to distinguish between:

- **Task-relevant linguistic signals:** Deceptive narratives, fraudulent intentions, persuasion strategies, and financial scam indicators.
- **Collection-specific characteristics:** Placeholder conventions, formatting patterns, source-specific headers, and other artifacts that may reflect how a dataset was collected or processed.

The experiments in this notebook identify the presence of source-related signals but do not isolate the exact contribution of each signal to DistilBERT's predictions.

### Relationship to the Primary Benchmark

The existing group-aware split prevents near-duplicate clusters from crossing the training, validation, and test partitions. All nine source families are represented in each partition.

Consequently, the Notebook 06 evaluation measures performance on held-out records from source families represented during training. It does not constitute an independent evaluation on entirely unseen data sources.

DistilBERT achieved 99.895% test accuracy on this benchmark, with a scam-class F1-score of 99.712%. These results remain valid for the evaluated test distribution, while their generalization to previously unseen source distributions remains an open question.

### Limitations and Future Evaluation

The following limitations should be considered when interpreting the findings:

- The collected sources have class-exclusive distributions, limiting the ability to separate source identity from class association using the existing corpus alone.
- Source identification performance does not directly measure the degree to which the Transformer relies on source-specific characteristics.
- The surface-feature and placeholder-only baselines are diagnostic experiments, not comprehensive explanations of the Transformer model.
- The controlled placeholder-wrapper experiment examined only a restricted transformation and should not be generalized to all preprocessing artifacts.
- The current benchmark does not establish cross-source generalization to entirely unseen collection sources.

Further evaluation could investigate source-held-out testing, source-wise Transformer performance, and the effect of synthetic scam data augmentation on binary classification.

### Conclusion

This investigation confirms that source-specific textual characteristics are present and recoverable in the assembled financial communication corpus. Simple structural and placeholder-based features provide useful classification signals, but neither independently accounts for the Transformer model's performance.

The findings do not invalidate the existing group-aware benchmark or demonstrate that DistilBERT relies on source identity as a shortcut. Instead, they establish important evaluation context and identify cross-source generalization as a separate research question.

The primary binary benchmark is therefore retained, with its source composition and generalization limitations explicitly acknowledged.
